
## 1 Setup del Notebook
## Lenguaje: Python
## Compute: Serverless (default)

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp, to_date
)

spark


## 2 Definimos el portfolio

In [0]:
portfolio = [
    ("SPY", "SPDR S&P 500 ETF", "ETF", "Index", "USD", 0.60, True),
    ("QQQ", "Invesco QQQ Trust", "ETF", "Technology", "USD", 0.20, True),
    ("GLD", "SPDR Gold Shares", "ETF", "Commodity", "USD", 0.10, True),
    ("BTC-USD", "Bitcoin", "Crypto", "Crypto", "USD", 0.10, True),
]


## 3 Creamos bronze_assets

En Bronze no pensamos en métricas ni en negocio.
Solo traemos datos, los versionamos y los dejamos listos para ser transformados.

In [0]:
assets_df = (
    spark.createDataFrame(
        portfolio,
        ["symbol", "asset_name", "asset_type", "sector", "currency", "weight_target", "is_active"]
    )
    .withColumn("created_at", current_timestamp())
)

assets_df.write.mode("overwrite").saveAsTable(
    "workspace.portfolio_intel.bronze_assets"
)


## 4 Descarga de precios (2 años)

In [0]:
%pip install yfinance


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import yfinance as yf
import pandas as pd


In [0]:
symbols = [row[0] for row in portfolio]

raw_prices = []

for symbol in symbols:
    df = yf.download(
        symbol,
        period="2y",
        interval="1d",
        auto_adjust=False,
        progress=False,
        multi_level_index=False
    )
    if not df.empty:
        df["symbol"] = symbol
        raw_prices.append(df.reset_index())


## 5 Unificamos y pasamos a Spark DataFrame




In [0]:
prices_pd = pd.concat(raw_prices, ignore_index=True)

prices_spark = (
    spark.createDataFrame(prices_pd)
    .withColumnRenamed("Date", "date")
    .withColumnRenamed("Open", "open")
    .withColumnRenamed("High", "high")
    .withColumnRenamed("Low", "low")
    .withColumnRenamed("Close", "close")
    .withColumnRenamed("Adj Close", "adj_close")
    .withColumnRenamed("Volume", "volume")
    .withColumn("date", to_date(col("date")))
    .withColumn("source", lit("yfinance"))
    .withColumn("ingestion_ts", current_timestamp())
)

display(prices_spark.limit(10))

date,adj_close,close,high,low,open,volume,symbol,source,ingestion_ts
2024-01-09,462.4477844238281,473.8800048828125,474.92999267578125,471.3500061035156,471.8699951171875,65931400,SPY,yfinance,2026-01-09T19:35:17.268Z
2024-01-10,465.0631103515625,476.55999755859375,477.45001220703125,473.8699951171875,474.1600036621094,67310600,SPY,yfinance,2026-01-09T19:35:17.268Z
2024-01-11,464.8581848144531,476.3500061035156,478.1199951171875,472.260009765625,477.5899963378906,77940700,SPY,yfinance,2026-01-09T19:35:17.268Z
2024-01-12,465.1802673339844,476.67999267578125,478.6000061035156,475.2300109863281,477.8399963378906,58026400,SPY,yfinance,2026-01-09T19:35:17.268Z
2024-01-16,463.47247314453125,474.92999267578125,476.6099853515625,473.05999755859375,475.260009765625,85014900,SPY,yfinance,2026-01-09T19:35:17.268Z
2024-01-17,460.8961181640625,472.2900085449219,472.7900085449219,469.8699951171875,471.82000732421875,68843900,SPY,yfinance,2026-01-09T19:35:17.268Z
2024-01-18,464.99481201171875,476.489990234375,477.05999755859375,472.4200134277344,474.010009765625,91856200,SPY,yfinance,2026-01-09T19:35:17.268Z
2024-01-19,470.79150390625,482.42999267578125,482.7200012207031,476.5400085449219,477.6499938964844,110834500,SPY,yfinance,2026-01-09T19:35:17.268Z
2024-01-22,471.78692626953125,483.45001220703125,485.2200012207031,482.7799987792969,484.010009765625,75844900,SPY,yfinance,2026-01-09T19:35:17.268Z
2024-01-23,473.16290283203125,484.8599853515625,485.1099853515625,482.8900146484375,484.010009765625,49945300,SPY,yfinance,2026-01-09T19:35:17.268Z


In [0]:
(
    prices_spark
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("workspace.portfolio_intel.bronze_prices")
)


In [0]:
%sql

SELECT symbol, COUNT(*) AS records
FROM workspace.portfolio_intel.bronze_prices
GROUP BY symbol
ORDER BY records DESC;


symbol,records
BTC-USD,732
GLD,503
QQQ,503
SPY,503


Si una tabla existe pero está vacía, casi siempre el error está en el write, no en el read.

Y más preciso aún:

En Unity Catalog, saveAsTable con nombre completo no es opcional, es obligatorio.

In [0]:
%sql
SELECT *
FROM workspace.portfolio_intel.bronze_assets;


symbol,asset_name,asset_type,sector,currency,weight_target,is_active,created_at
SPY,SPDR S&P 500 ETF,ETF,Index,USD,0.6,true,2026-01-09T19:34:36.302Z
QQQ,Invesco QQQ Trust,ETF,Technology,USD,0.2,true,2026-01-09T19:34:36.302Z
GLD,SPDR Gold Shares,ETF,Commodity,USD,0.1,true,2026-01-09T19:34:36.302Z
BTC-USD,Bitcoin,Crypto,Crypto,USD,0.1,true,2026-01-09T19:34:36.302Z


In [0]:
spark.sql("SHOW TABLES IN workspace.portfolio_intel").show()


+---------------+--------------------+-----------+
|       database|           tableName|isTemporary|
+---------------+--------------------+-----------+
|portfolio_intel|       bronze_assets|      false|
|portfolio_intel|       bronze_prices|      false|
|portfolio_intel|gold_asset_contri...|      false|
|portfolio_intel|gold_portfolio_daily|      false|
|portfolio_intel|gold_return_predi...|      false|
|portfolio_intel|   gold_risk_metrics|      false|
|portfolio_intel|  silver_features_ml|      false|
|portfolio_intel| silver_prices_clean|      false|
|portfolio_intel|      silver_returns|      false|
+---------------+--------------------+-----------+



In [0]:
spark.sql("SELECT COUNT(*) FROM workspace.portfolio_intel.bronze_prices").show()


+--------+
|COUNT(*)|
+--------+
|    2241|
+--------+



In [0]:
%sql
SHOW TABLES IN workspace.portfolio_intel


database,tableName,isTemporary
portfolio_intel,bronze_assets,false
portfolio_intel,bronze_prices,false
portfolio_intel,gold_asset_contribution,false
portfolio_intel,gold_portfolio_daily,false
portfolio_intel,gold_return_predictions,false
portfolio_intel,gold_risk_metrics,false
portfolio_intel,silver_features_ml,false
portfolio_intel,silver_prices_clean,false
portfolio_intel,silver_returns,false


In [0]:
%sql
SELECT COUNT(*) FROM workspace.portfolio_intel.bronze_prices

COUNT(*)
0
